# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, exploration, and basic processing of the FAIR² Mental Health survey dataset using the `mlcroissant` library.

### Dataset Source
This dataset is published with a Croissant schema, available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema provides programmatic access to rich metadata, record sets, and documentation for robust AI-ready data exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic overview (accessed via object attribute)
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs from the Croissant schema.

In `mlcroissant`, record sets, fields, and columns are referenced by their `@id`. For robust analysis, this section extracts schema information about record sets and their structure.

In [ ]:
# Get all record sets and their @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- {rs['@id']} | {rs.get('name', 'Unnamed')}")

# For each record set, list fields (@id) and columns/@id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', 'Unnamed')})")
    fields = rs.get('fields', [])
    print("  Fields:")
    for f in fields:
        print(f"    - {f['@id']} | {f.get('name', f['@id'])} (dataType: {f.get('dataType', '')})")
    columns = rs.get('columns', [])
    print("  Columns:")
    for c in columns:
        print(f"    - {c['@id']} | {c.get('name', c['@id'])} (dataType: {c.get('dataType', '')})")

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis.

Use the record set and field `@id`s discovered above. For demonstration, we'll use the primary survey record set, typically named or described as the main data table.


In [ ]:
# List all record set @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load each record set into a dataframe
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded Record Set: {rs_id}, Records: {len(df)}")
    except Exception as e:
        print(f"Failed to load {rs_id}: {e}")

# Choose the main record set (example, replace with the primary @id from the overview if known)
main_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_rs_id:
    print(f"\nColumns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()
else:
    print("No record sets detected.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records based on specific criteria, normalize numeric fields, and categorize/group records.

**All fields are referenced by their `@id`.**

In [ ]:
# Pick a numeric field for analysis - choose by @id
# Update to match an actual numeric field @id from step 2 (example: 'gad_7_score' or similar)
numeric_field_id = 'gad_7_total'  # Example; adjust to correct @id found above
record_set_id = main_rs_id

# If field exists, proceed
if record_set_id and numeric_field_id in dataframes[record_set_id].columns:
    threshold = 10
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    group_field_id = 'education_level'  # Adjust to correct @id to group by, e.g. 'level_of_education'
    if group_field_id in dataframes[record_set_id].columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field_id}' not found in record set '{record_set_id}'. Check available columns.")

## 5. Visualization
Visualize distributions or relationships between measurements, referencing fields by their `@id`.


In [ ]:
# Example: Histogram of the numeric field
if record_set_id and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8, 4))
    plt.hist(df[numeric_field_id].dropna(), bins=15, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Example: Boxplot by group field
if record_set_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(10, 5))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle("")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook loaded the FAIR² Mental Health Survey Dataset from Kilifi County using `mlcroissant`, reviewed its Croissant schema with references by `@id`, extracted data into Pandas DataFrames, and conducted basic filtering, normalization, grouping, and visualization.

**Summary:**
- Data accessed programmatically via Croissant schema ensures reproducibility and traceability.
- Survey numeric scores and demographic fields can be directly selected and analyzed using their `@id`.
- The workflow is ready for further AI analysis such as clustering, predictive modeling, or additional data cleaning.

For more advanced operations, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant/blob/main/python/mlcroissant/README.md) and the full Croissant schema for the dataset.